# Constructing a Goal-Based Agent using an LLM

Everything for this lab in one notebook: the write up, the code, the runs and the tests. Run all cells from top to bottom.

## Task 1: Understanding the Problem

1. **Environment:** a 7 x 21 grid warehouse. Cells are either free (`.`, `S`, `G`) or shelving (`#`). It is fully observable, deterministic, static and discrete.
2. **Goal:** reach the dispatch cell `G` at (row 1, col 19) starting from `S` at (1, 1), without entering a `#` cell.
3. **Actions:** Up, Down, Left, Right. Each moves exactly one square.
4. **Information the agent maintains:** its current position, the map (which cells are blocked), the goal position, and while planning, the set of positions already explored plus a parent pointer for each one so the path can be rebuilt.
5. **Why goal-based, not reflex:** a reflex agent maps the current percept straight to an action ("if wall ahead, turn right"). That fails here because the right move depends on where the goal is and what lies several squares ahead. For example, from S the agent must first go right and then dip down to row 2 to get around the shelf at column 6, which no single local rule captures. The goal-based agent asks "which sequence of actions ends in the goal state?" and searches for it.

**Think about it (warehouse twice as large):** BFS still works and is still optimal, but the number of states it may expand grows with the area of the grid (doubling both sides gives about 4x the cells), and memory grows the same way. On big maps an informed search such as A* with Manhattan distance would usually expand fewer states. Extra difficulties at scale: moving obstacles (other vehicles) make the environment dynamic so the plan needs replanning, multiple robots need coordination, and turning or acceleration costs mean "one move = cost 1" no longer holds.

## Task 2: Agent Design

| Component | Design |
|---|---|
| Environment | `Environment` class: grid, `is_free(pos)`, transition `result(pos, action)` |
| Current state | `(row, col)` tuple stored in `agent.state` |
| Goal | `agent.goal`, tested by `goal_test(pos)` |
| Actions | `ACTIONS` dict mapping name to `(dr, dc)`; `legal_actions` filters out walls |
| Decision making | `search()` runs BFS and returns an action list, `run()` executes it |

```mermaid
flowchart LR
    E[Environment: warehouse grid] -->|percept: position, map| S[Current state]
    S --> M[Model: how actions change position]
    G[Goal: reach G] --> D[Decision: BFS search]
    M --> D
    S --> D
    D -->|next action| A[Actuators: Up/Down/Left/Right]
    A --> E
```

## Task 3: Prompt Engineering

Prompt used (based on the suggested prompt, with my design added):

> Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem below. [map pasted]. Represent the warehouse as a 2D grid. The agent has a state (row, col), a goal (the G cell), actions Up/Down/Left/Right that move one square, and a decision component that searches for an action sequence. Find a collision-free path from S to G, avoid all obstacles, print the path found or a message if none exists, and explain the search algorithm you chose and why it is appropriate. Use only the standard library. Also give me unit tests including a case where no path exists.

Results:

```
Path length: 20 moves
States expanded: 59
Actions: Right Right Right Down Right Right Right Up Right Right Right Right Right Right Right Right Right Right Right Right

#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################
```

**1. Working on the first attempt?** Yes. The program ran first time and all five unit tests passed. I still checked the output by hand: 20 moves is the minimum, because the Manhattan distance from S to G is 18 and the shelf at column 6 of row 1 forces a detour of one step down and one step up.

**2. How could the prompt be improved?** Specifying the state, goal and action set explicitly (rather than just "write a program") stopped the LLM from inventing diagonal moves. Asking for tests in the same prompt was also useful. A further improvement would be to request that it reports the number of states expanded, so different algorithms can be compared later.

**3. Which algorithm?** Breadth-First Search.

**4. Why?** Every move costs the same, so BFS returns a shortest path, it is complete, and on a 147-cell grid its memory cost is negligible. It is also the simplest algorithm that is both correct and optimal here, which is what an LLM tends to reach for when the prompt gives no heuristic. A* would also be optimal but adds complexity that this map does not need.

### Implementation

In [1]:
"""
Lab 1: Goal-Based Agent for Warehouse Navigation.

The agent keeps an internal model of the warehouse (the map), knows its goal
(the dispatch cell G), and chooses actions that move it towards that goal.
Decision making is done by Breadth-First Search (BFS):

* every move costs the same (1 square), so BFS returns a shortest path;
* the grid is small, so the extra memory BFS uses is not a problem;
* BFS is complete: if any path exists, it finds one.

"""

from collections import deque

WAREHOUSE = """\
#####################
#S....#............G#
#.##....##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################"""

# action name -> (row change, column change)
ACTIONS = {
    "Up": (-1, 0),
    "Down": (1, 0),
    "Left": (0, -1),
    "Right": (0, 1),
}


class Environment:
    """The warehouse world. Knows walls, start and goal."""

    def __init__(self, text):
        self.grid = [list(line) for line in text.strip("\n").splitlines()]
        self.rows = len(self.grid)
        self.cols = len(self.grid[0])
        self.start = self._find("S")
        self.goal = self._find("G")

    def _find(self, symbol):
        for r, row in enumerate(self.grid):
            for c, ch in enumerate(row):
                if ch == symbol:
                    return (r, c)
        raise ValueError(f"Symbol {symbol!r} not found in map")

    def is_free(self, pos):
        r, c = pos
        return 0 <= r < self.rows and 0 <= c < self.cols and self.grid[r][c] != "#"

    def result(self, pos, action):
        """Transition model: where does an action take the vehicle?"""
        dr, dc = ACTIONS[action]
        return (pos[0] + dr, pos[1] + dc)


class GoalBasedAgent:
    """
    Goal-based agent:
      state     -> current position (r, c)
      goal      -> target position
      actions   -> Up / Down / Left / Right
      decision  -> search for an action sequence that reaches the goal
    """

    def __init__(self, env):
        self.env = env
        self.state = env.start
        self.goal = env.goal
        self.plan = []
        self.states_expanded = 0

    def goal_test(self, pos):
        return pos == self.goal

    def legal_actions(self, pos):
        return [a for a in ACTIONS if self.env.is_free(self.env.result(pos, a))]

    def search(self):
        """BFS from the current state. Returns a list of actions or None."""
        frontier = deque([self.state])
        parent = {self.state: (None, None)}  # also acts as the visited set
        self.states_expanded = 0

        while frontier:
            pos = frontier.popleft()
            self.states_expanded += 1
            if self.goal_test(pos):
                return self._reconstruct(parent, pos)
            for action in self.legal_actions(pos):
                nxt = self.env.result(pos, action)
                if nxt not in parent:
                    parent[nxt] = (pos, action)
                    frontier.append(nxt)
        return None

    @staticmethod
    def _reconstruct(parent, pos):
        actions = []
        while parent[pos][0] is not None:
            prev, action = parent[pos]
            actions.append(action)
            pos = prev
        return list(reversed(actions))

    def run(self):
        """Plan, then execute the plan step by step (perceive, decide, act)."""
        self.plan = self.search()
        if self.plan is None:
            return None
        trajectory = [self.state]
        for action in self.plan:
            self.state = self.env.result(self.state, action)
            assert self.env.is_free(self.state), "Agent walked into a shelf!"
            trajectory.append(self.state)
        return trajectory


def draw_path(env, trajectory):
    grid = [row[:] for row in env.grid]
    for r, c in trajectory[1:-1]:
        grid[r][c] = "*"
    return "\n".join("".join(row) for row in grid)


def run_agent(text=WAREHOUSE):
    env = Environment(text)
    agent = GoalBasedAgent(env)
    trajectory = agent.run()
    if trajectory is None:
        print("No collision-free path exists from S to G.")
        return None
    print(f"Start: {env.start}  Goal: {env.goal}")
    print(f"Path length: {len(agent.plan)} moves")
    print(f"States expanded: {agent.states_expanded}")
    print("Actions:", " ".join(agent.plan))
    print("Positions:", " -> ".join(str(p) for p in trajectory))
    print()
    print(draw_path(env, trajectory))
    return agent.plan

### Run on the warehouse map

In [2]:
run_agent()

Start: (1, 1)  Goal: (1, 19)
Path length: 20 moves
States expanded: 59
Actions: Right Right Right Down Right Right Right Up Right Right Right Right Right Right Right Right Right Right Right Right
Positions: (1, 1) -> (1, 2) -> (1, 3) -> (1, 4) -> (2, 4) -> (2, 5) -> (2, 6) -> (2, 7) -> (1, 7) -> (1, 8) -> (1, 9) -> (1, 10) -> (1, 11) -> (1, 12) -> (1, 13) -> (1, 14) -> (1, 15) -> (1, 16) -> (1, 17) -> (1, 18) -> (1, 19)

#####################
#S***.#************G#
#.##****##########..#
#....##.............#
#.######.###.#.###..#
#........#..........#
#####################


['Right',
 'Right',
 'Right',
 'Down',
 'Right',
 'Right',
 'Right',
 'Up',
 'Right',
 'Right',
 'Right',
 'Right',
 'Right',
 'Right',
 'Right',
 'Right',
 'Right',
 'Right',
 'Right',
 'Right']

### Unit tests

In [3]:
"""Unit tests for the Lab 1 agent.  Run: python -m unittest -v"""
import unittest


def solve(text):
    agent = GoalBasedAgent(Environment(text))
    return agent, agent.run()


class TestAgent(unittest.TestCase):
    def test_original_map_finds_path(self):
        agent, traj = solve(WAREHOUSE)
        self.assertIsNotNone(traj)
        self.assertEqual(traj[-1], agent.goal)

    def test_path_never_hits_obstacle(self):
        agent, traj = solve(WAREHOUSE)
        for pos in traj:
            self.assertTrue(agent.env.is_free(pos))

    def test_moves_are_single_steps(self):
        _, traj = solve(WAREHOUSE)
        for (r1, c1), (r2, c2) in zip(traj, traj[1:]):
            self.assertEqual(abs(r1 - r2) + abs(c1 - c2), 1)

    def test_adjacent_goal(self):
        agent, traj = solve("#####\n#SG##\n#####")
        self.assertEqual(agent.plan, ["Right"])

    def test_no_path(self):
        agent, traj = solve("#######\n#S.#..#\n###.#G#\n#######")
        self.assertIsNone(traj)


unittest.main(argv=["notebook"], exit=False, verbosity=2)

test_adjacent_goal (__main__.TestAgent.test_adjacent_goal) ... 

ok


test_moves_are_single_steps (__main__.TestAgent.test_moves_are_single_steps) ... 

ok


test_no_path (__main__.TestAgent.test_no_path) ... 

...

test_original_map_finds_path (__main__.TestAgent.test_original_map_finds_path) ... 

ok


test_path_never_hits_obstacle (__main__.TestAgent.test_path_never_hits_obstacle) ... 

ok


----------------------------------------------------------------------
Ran 5 tests in 0.011s

OK
